# TTC Delay Dataset Analysis Template

This notebook analyzes one TTC delay dataset file (CSV or Excel, even if extension is wrong).

How to use:
1. Change `FILE_PATH` in the next code cell.
2. Run all cells.
3. Review the generated tables and charts for insights.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Change only this file path when you want to analyze a different year/file
FILE_PATH = Path('../data/ttc_bus_delay_2014.csv')

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')


In [ ]:
COLUMN_ALIASES = {
    'date': 'date',
    'report_date': 'date',
    'incident_date': 'date',
    'time': 'time',
    'report_time': 'time',
    'incident_time': 'time',
    'time_of_day': 'time',
    'route': 'route',
    'route_number': 'route',
    'route_name': 'route',
    'route_no': 'route',
    'route_num': 'route',
    'line': 'route',
    'location': 'location',
    'location_name': 'location',
    'intersection': 'location',
    'incident': 'incident',
    'incident_type': 'incident',
    'incident_description': 'incident',
    'delay_reason': 'incident',
    'min_delay': 'min_delay',
    'min delay': 'min_delay',
    'delay_minutes': 'min_delay',
    'delay': 'min_delay',
    'min_gap': 'min_gap',
    'min gap': 'min_gap',
    'gap_minutes': 'min_gap',
    'gap': 'min_gap',
    'day': 'day',
    'direction': 'direction',
    'vehicle': 'vehicle',
}

REQUIRED = ['date', 'time', 'route', 'location', 'incident', 'min_delay', 'min_gap']

def normalize_col(col: str) -> str:
    c = str(col).strip().strip('\"').replace('﻿', '').replace('​', '')
    c = c.replace('/', '_').replace('-', '_').lower()
    c = re.sub(r'[^a-z0-9_]+', '_', c)
    c = re.sub(r'_+', '_', c).strip('_')
    return COLUMN_ALIASES.get(c, c)

def read_any_table(file_path: Path) -> pd.DataFrame:
    payload = file_path.read_bytes()

    # xlsx files are zip containers starting with PK
    is_xlsx_payload = payload[:2] == b'PK'
    suffix = file_path.suffix.lower()

    if suffix in {'.xlsx', '.xls'} or is_xlsx_payload:
        return pd.read_excel(file_path)

    encodings = ('utf-8', 'utf-8-sig', 'utf-16', 'cp1252', 'latin1')
    seps = (None, ',', '\t', ';', '|')

    for enc in encodings:
        for sep in seps:
            try:
                return pd.read_csv(file_path, encoding=enc, sep=sep, engine='python', on_bad_lines='skip')
            except Exception:
                pass

    raise ValueError(f'Could not read file: {file_path}')

def standardize(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [normalize_col(c) for c in out.columns]

    # Build missing core columns with fuzzy fallbacks
    for target, patterns in {
        'date': ('date',),
        'time': ('time',),
        'route': ('route', 'line'),
        'location': ('location', 'intersection', 'stop'),
        'incident': ('incident', 'reason', 'cause'),
        'min_delay': ('delay',),
        'min_gap': ('gap',),
    }.items():
        if target in out.columns:
            continue
        match = next((c for c in out.columns if any(p in c for p in patterns)), None)
        if match is not None:
            out[target] = out[match]

    # If min_gap is absent, use min_delay as a fallback
    if 'min_gap' not in out.columns and 'min_delay' in out.columns:
        out['min_gap'] = out['min_delay']

    missing = [c for c in REQUIRED if c not in out.columns]
    if missing:
        raise ValueError(f'Missing required columns after standardization: {missing}')

    # Parse date/time and numeric fields
    out['date_parsed'] = pd.to_datetime(out['date'], errors='coerce')

    # Attempt to parse time directly; if fails, derive from date if datetime-like
    parsed_time = pd.to_datetime(out['time'].astype(str), errors='coerce')
    out['time_parsed'] = parsed_time.dt.time

    out['min_delay'] = pd.to_numeric(out['min_delay'], errors='coerce')
    out['min_gap'] = pd.to_numeric(out['min_gap'], errors='coerce')

    # Derive time-based dimensions for analysis
    out['year'] = out['date_parsed'].dt.year
    out['month'] = out['date_parsed'].dt.month
    out['day_name'] = out['date_parsed'].dt.day_name()

    hour_from_time = pd.to_datetime(out['time'].astype(str), errors='coerce').dt.hour
    hour_from_date = out['date_parsed'].dt.hour
    out['hour'] = hour_from_time.fillna(hour_from_date)

    return out


In [ ]:
if not FILE_PATH.exists():
    raise FileNotFoundError(f'File not found: {FILE_PATH.resolve()}')

raw_df = read_any_table(FILE_PATH)
df = standardize(raw_df)

print('Analyzed file:', FILE_PATH.resolve())
print('Raw shape:', raw_df.shape)
print('Standardized shape:', df.shape)

df.head()


In [ ]:
# Data quality and descriptive statistics
missing_table = (
    df[REQUIRED]
    .isna()
    .sum()
    .rename('missing_count')
    .to_frame()
    .assign(missing_pct=lambda t: (100 * t['missing_count'] / len(df)).round(2))
)

display(missing_table)
display(df[['min_delay', 'min_gap']].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))


In [ ]:
# Top incidents and routes by delay impact
top_incidents = (
    df.dropna(subset=['incident'])
      .groupby('incident', as_index=False)
      .agg(events=('incident', 'size'), total_delay=('min_delay', 'sum'), avg_delay=('min_delay', 'mean'))
      .sort_values('total_delay', ascending=False)
      .head(10)
)

top_routes = (
    df.dropna(subset=['route'])
      .groupby('route', as_index=False)
      .agg(events=('route', 'size'), total_delay=('min_delay', 'sum'), avg_delay=('min_delay', 'mean'))
      .sort_values('total_delay', ascending=False)
      .head(10)
)

display(top_incidents)
display(top_routes)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=top_incidents, y='incident', x='total_delay', ax=axes[0], color='#2f7d6d')
axes[0].set_title('Top 10 Incidents by Total Delay Minutes')
axes[0].set_xlabel('Total delay (minutes)')
axes[0].set_ylabel('Incident')

sns.barplot(data=top_routes, y='route', x='total_delay', ax=axes[1], color='#cc7a00')
axes[1].set_title('Top 10 Routes by Total Delay Minutes')
axes[1].set_xlabel('Total delay (minutes)')
axes[1].set_ylabel('Route')

plt.tight_layout()
plt.show()


In [ ]:
# Time-based patterns
hourly = (
    df.dropna(subset=['hour'])
      .groupby('hour', as_index=False)
      .agg(events=('hour', 'size'), avg_delay=('min_delay', 'mean'))
      .sort_values('hour')
)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = (
    df.dropna(subset=['day_name'])
      .groupby('day_name', as_index=False)
      .agg(events=('day_name', 'size'), avg_delay=('min_delay', 'mean'))
)
daily['day_name'] = pd.Categorical(daily['day_name'], categories=day_order, ordered=True)
daily = daily.sort_values('day_name')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=hourly, x='hour', y='events', marker='o', ax=axes[0], color='#1f77b4')
axes[0].set_title('Delay Events by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Number of delay events')

sns.barplot(data=daily, x='day_name', y='events', ax=axes[1], color='#9c6644')
axes[1].set_title('Delay Events by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Number of delay events')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()


In [ ]:
# Top locations table and key numeric insights
top_locations = (
    df.dropna(subset=['location'])
      .groupby('location', as_index=False)
      .agg(events=('location', 'size'), total_delay=('min_delay', 'sum'), avg_delay=('min_delay', 'mean'))
      .sort_values('total_delay', ascending=False)
      .head(15)
)
display(top_locations)

valid_delay = df['min_delay'].dropna()
if len(valid_delay) > 0:
    print(f'Total records: {len(df):,}')
    print(f'Total delay minutes: {valid_delay.sum():,.0f}')
    print(f'Average delay minutes: {valid_delay.mean():.2f}')
    print(f'Median delay minutes: {valid_delay.median():.2f}')
    print(f'95th percentile delay minutes: {valid_delay.quantile(0.95):.2f}')
else:
    print('No valid min_delay values found.')
